# PPO for RLHF: Reinforcement Learning from Human Feedback

This notebook covers **PPO (Proximal Policy Optimization)** applied to RLHF for aligning language models with human preferences.

We'll cover:
1. **RLHF Pipeline Overview**: SFT → Reward Model → PPO
2. **PPO Theory**: Clipped surrogate objective, GAE, value function
3. **Reward Model**: Training on preference data
4. **PPO Implementation**: Actor-Critic from scratch
5. **RLHF-specific additions**: KL penalty, reward normalization
6. **Practical RLHF with TRL**: Using `PPOTrainer`
7. **Monitoring**: Reward, KL divergence, policy entropy

## The Full RLHF Pipeline

```
Stage 1: SFT (Supervised Fine-Tuning)
   Pre-trained LLM → Fine-tune on demonstrations → π_SFT

Stage 2: Reward Model Training
   (prompt, chosen, rejected) → Train r_φ(x, y) to match human preferences

Stage 3: PPO Optimization
   Maximize: E[r_φ(x, y)] - β·KL[π_θ || π_SFT]
   Using: PPO algorithm with the reward model as environment
```

**Key difference from DPO**: PPO uses the reward model **online** — the policy generates new samples each step, gets scored, and updates via RL. This allows the policy to explore new behaviors beyond the training data.

## 1. Install and Import Libraries

In [ ]:
# Install required libraries (run once)
# !pip install transformers>=4.40.0 peft>=0.10.0 trl>=0.8.6 datasets>=2.18.0 \
#              accelerate>=0.28.0 bitsandbytes>=0.43.0

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. PPO Theory

### Standard Policy Gradient (REINFORCE)

The simplest policy gradient objective:
$$\mathcal{L}_{\text{PG}} = \mathbb{E}_t \left[ \log \pi_\theta(a_t | s_t) \cdot A_t \right]$$

Where $A_t$ is the **advantage** (how much better action $a_t$ is vs. average).

**Problem**: Large policy updates cause instability. New policy can be very different from old.

### PPO: Clipped Surrogate Objective

PPO constrains updates using importance sampling ratio $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$:

$$\mathcal{L}_{\text{PPO}} = \mathbb{E}_t \left[ \min\left( r_t(\theta) A_t,\ \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) A_t \right) \right]$$

**Intuition**:
- If $A_t > 0$ (good action): increase $\pi_\theta$, but not by more than $1+\epsilon$
- If $A_t < 0$ (bad action): decrease $\pi_\theta$, but not by more than $1-\epsilon$

### Generalized Advantage Estimation (GAE)

GAE balances bias vs variance in advantage estimation:
$$A_t^{\text{GAE}} = \sum_{l=0}^{\infty} (\gamma \lambda)^l \delta_{t+l}$$

Where $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ is the TD error.

### RLHF-Specific Objective

For language models, the "reward" includes a KL penalty:
$$r(x, y) = r_\phi(x, y) - \beta \cdot \underbrace{\log \frac{\pi_\theta(y|x)}{\pi_{\text{SFT}}(y|x)}}_{\text{KL from SFT model}}$$

In [ ]:
# Visualize PPO clipping mechanism
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epsilon = 0.2
ratios = np.linspace(0, 2.5, 300)
ratio_tensor = torch.tensor(ratios)

# Plot 1: PPO clipping for positive and negative advantages
ax = axes[0]
for A, color, label in [(1.0, 'green', 'A > 0 (good action)'), (-1.0, 'red', 'A < 0 (bad action)')]:
    surrogate = ratio_tensor * A
    clipped = torch.clamp(ratio_tensor, 1 - epsilon, 1 + epsilon) * A
    ppo = torch.min(surrogate, clipped)
    
    ax.plot(ratios, surrogate.numpy(), '--', color=color, alpha=0.5, linewidth=1.5, label=f'r·A (A={A})')
    ax.plot(ratios, ppo.numpy(), '-', color=color, linewidth=2.5, label=f'PPO (A={A})')

ax.axvline(1.0, color='black', linestyle=':', alpha=0.7, label='r=1 (no change)')
ax.axvline(1 - epsilon, color='gray', linestyle='--', alpha=0.5, label=f'clip bounds [{1-epsilon}, {1+epsilon}]')
ax.axvline(1 + epsilon, color='gray', linestyle='--', alpha=0.5)
ax.axhline(0, color='black', linewidth=0.5, alpha=0.5)

ax.fill_betweenx([-1.5, 1.5], 1-epsilon, 1+epsilon, alpha=0.1, color='blue', label='Clipping region')

ax.set_xlabel('Probability Ratio r(θ) = π_θ / π_old')
ax.set_ylabel('PPO Objective')
ax.set_title(f'PPO Clipped Surrogate (ε={epsilon})')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_ylim([-1.5, 1.5])
ax.set_xlim([0, 2.5])

# Plot 2: Effect of clipping on update magnitude
ax = axes[1]
epsilons = [0.1, 0.2, 0.3, 0.5]
colors = ['#e74c3c', '#e67e22', '#3498db', '#9b59b6']

for eps, color in zip(epsilons, colors):
    A = 1.0
    clipped = torch.clamp(ratio_tensor, 1 - eps, 1 + eps) * A
    ppo = torch.min(ratio_tensor * A, clipped)
    ax.plot(ratios, ppo.numpy(), color=color, linewidth=2, label=f'ε={eps}')

ax.axvline(1.0, color='black', linestyle=':', alpha=0.7)
ax.axhline(1.0, color='black', linestyle=':', alpha=0.5)
ax.set_xlabel('Probability Ratio r(θ)')
ax.set_ylabel('PPO Objective (A > 0)')
ax.set_title('PPO Clipping: Effect of ε on Update Size')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 2.5])

plt.tight_layout()
plt.savefig('ppo_clipping.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. Reward Model

The reward model $r_\phi(x, y)$ maps (prompt, response) pairs to scalar rewards.

It's typically a **language model with a scalar head** trained on preference data using the Bradley-Terry loss:

$$\mathcal{L}_{\text{RM}} = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma(r_\phi(x, y_w) - r_\phi(x, y_l)) \right]$$

In [ ]:
class RewardModel(nn.Module):
    """
    Reward model: LM backbone + scalar value head.
    
    Takes (prompt + response) tokens → outputs scalar reward.
    Trained on preference pairs using Bradley-Terry loss.
    """
    
    def __init__(self, backbone: nn.Module, hidden_size: int, dropout: float = 0.1):
        super().__init__()
        self.backbone = backbone
        # Value head: projects last hidden state to scalar reward
        self.value_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Linear(hidden_size // 2, 1),
        )
    
    def get_last_hidden_state(self, input_ids: torch.Tensor) -> torch.Tensor:
        """Get the hidden state at the last (non-padding) position."""
        hidden = self.backbone(input_ids)  # (batch, seq, hidden)
        # Use last token's hidden state as the sequence representation
        return hidden[:, -1, :]  # (batch, hidden)
    
    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """Returns scalar reward for each input. Shape: (batch,)"""
        last_hidden = self.get_last_hidden_state(input_ids)
        reward = self.value_head(last_hidden).squeeze(-1)  # (batch,)
        return reward


def reward_model_loss(
    reward_chosen: torch.Tensor,
    reward_rejected: torch.Tensor,
    margin: float = 0.0,
) -> torch.Tensor:
    """
    Bradley-Terry reward model loss.
    
    Optionally with a margin: r(chosen) - r(rejected) > margin
    """
    if margin > 0:
        # Margin loss: encourage larger gap between chosen and rejected
        return -F.logsigmoid(reward_chosen - reward_rejected - margin).mean()
    return -F.logsigmoid(reward_chosen - reward_rejected).mean()


# Toy backbone for demonstration
class TinyBackbone(nn.Module):
    def __init__(self, vocab_size=200, hidden_size=64, num_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers=num_layers, batch_first=True)
    
    def forward(self, input_ids):
        x = self.embed(input_ids)
        out, _ = self.lstm(x)
        return out  # (batch, seq, hidden)


# Train a toy reward model
vocab_size = 200
hidden_size = 64
seq_len = 30
batch_size = 8

backbone = TinyBackbone(vocab_size=vocab_size, hidden_size=hidden_size)
rm = RewardModel(backbone, hidden_size=hidden_size)
rm_optimizer = torch.optim.AdamW(rm.parameters(), lr=1e-3)

# Simulate training: chosen responses get higher reward
rm_losses = []
rm_accuracies = []

print("Training Reward Model:")
print(f"{'Step':>6} {'Loss':>8} {'Accuracy':>10}")
print("-" * 28)

for step in range(50):
    chosen = torch.randint(0, vocab_size // 2, (batch_size, seq_len))     # High-quality tokens
    rejected = torch.randint(vocab_size // 2, vocab_size, (batch_size, seq_len))  # Low-quality tokens
    
    r_chosen = rm(chosen)
    r_rejected = rm(rejected)
    
    loss = reward_model_loss(r_chosen, r_rejected)
    accuracy = (r_chosen > r_rejected).float().mean().item()
    
    rm_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(rm.parameters(), 1.0)
    rm_optimizer.step()
    
    rm_losses.append(loss.item())
    rm_accuracies.append(accuracy)
    
    if step % 10 == 0:
        print(f"{step:>6} {loss.item():>8.4f} {accuracy:>10.4f}")

print(f"\nFinal reward model accuracy: {np.mean(rm_accuracies[-10:]):.4f}")

## 4. PPO Actor-Critic for Language Models

In the RLHF PPO setup:
- **Actor** ($\pi_\theta$): The language model policy — generates text
- **Critic** ($V_\psi$): Value function — estimates expected future reward from state $s_t$ (prefix)
- **Reference** ($\pi_{\text{SFT}}$): Frozen SFT model — used for KL penalty
- **Reward model** ($r_\phi$): Scores completed responses

### Token-Level vs Sequence-Level Rewards

In text generation:
- The **environment** only gives reward at the **end** of generation
- TRL distributes the reward: penalize each token with KL, give full reward to last token

In [ ]:
class ValueHead(nn.Module):
    """
    Value function head added on top of a language model.
    Estimates V(s_t) = expected future reward from the current sequence prefix.
    """
    def __init__(self, hidden_size: int, dropout: float = 0.1):
        super().__init__()
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Linear(hidden_size // 2, 1),
        )
    
    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        """
        Args:
            hidden_states: (batch, seq_len, hidden_size)
        Returns:
            values: (batch, seq_len) — value estimate at each token position
        """
        return self.head(hidden_states).squeeze(-1)


def compute_advantages_and_returns(
    rewards: torch.Tensor,
    values: torch.Tensor,
    masks: torch.Tensor,
    gamma: float = 0.99,
    gae_lambda: float = 0.95,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Compute Generalized Advantage Estimation (GAE) advantages and returns.
    
    A_t^GAE = Σ_l (γλ)^l δ_{t+l}
    δ_t = r_t + γ V(s_{t+1}) - V(s_t)  (TD error)
    
    Args:
        rewards: Per-token rewards (batch, seq_len)
        values: Value function estimates (batch, seq_len)
        masks: 1 for valid tokens, 0 for padding (batch, seq_len)
        gamma: Discount factor
        gae_lambda: GAE lambda (0=high bias/low var, 1=low bias/high var)
    
    Returns:
        advantages: GAE advantages (batch, seq_len)
        returns: Discounted returns (batch, seq_len) used for value function target
    """
    batch_size, seq_len = rewards.shape
    advantages = torch.zeros_like(rewards)
    
    # Backward pass through time
    last_gae_lam = 0
    for t in reversed(range(seq_len)):
        if t == seq_len - 1:
            next_value = 0  # Terminal state
        else:
            next_value = values[:, t + 1]
        
        # TD error: δ_t = r_t + γ·V(s_{t+1}) - V(s_t)
        delta = rewards[:, t] + gamma * next_value - values[:, t]
        delta = delta * masks[:, t]
        
        # GAE: A_t = δ_t + γλ·A_{t+1}
        last_gae_lam = delta + gamma * gae_lambda * last_gae_lam
        advantages[:, t] = last_gae_lam
    
    # Returns = Advantages + Values (used as value function targets)
    returns = advantages + values
    
    # Normalize advantages (stabilizes training)
    valid_advantages = advantages[masks.bool()]
    if valid_advantages.numel() > 1:
        mean = valid_advantages.mean()
        std = valid_advantages.std() + 1e-8
        advantages = (advantages - mean) / std
    
    return advantages, returns


# Test GAE computation
batch_size, seq_len = 2, 8

# Simulate: reward only at last token (typical for RLHF)
rewards = torch.zeros(batch_size, seq_len)
rewards[:, -1] = torch.tensor([1.5, 0.3])  # End-of-sequence rewards

values = torch.ones(batch_size, seq_len) * 0.5  # Value estimates
masks = torch.ones(batch_size, seq_len)

advantages, returns = compute_advantages_and_returns(
    rewards, values, masks, gamma=0.99, gae_lambda=0.95
)

print("GAE Computation Example:")
print(f"Rewards (batch 0):    {rewards[0].tolist()}")
print(f"Values (batch 0):     {[f'{v:.3f}' for v in values[0].tolist()]}")
print(f"Advantages (batch 0): {[f'{a:.3f}' for a in advantages[0].tolist()]}")
print(f"Returns (batch 0):    {[f'{r:.3f}' for r in returns[0].tolist()]}")
print(f"\nAdvantage for better response (reward=1.5): {advantages[0, -1]:.3f}")
print(f"Advantage for worse response (reward=0.3):  {advantages[1, -1]:.3f}")

In [ ]:
# Visualize GAE propagation backward through sequence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Simulate sparse reward (only at end) with different lambda values
seq_len = 20
terminal_reward = 2.0
gamma = 0.99

ax = axes[0]
for lam, color in [(0.0, '#e74c3c'), (0.5, '#e67e22'), (0.9, '#3498db'), (0.95, '#2ecc71'), (1.0, '#9b59b6')]:
    # Compute GAE with flat value function (V=0 everywhere)
    rewards_sim = np.zeros(seq_len)
    rewards_sim[-1] = terminal_reward
    values_sim = np.zeros(seq_len)
    
    advantages_sim = np.zeros(seq_len)
    last_gae = 0
    for t in reversed(range(seq_len)):
        next_val = values_sim[t+1] if t < seq_len-1 else 0
        delta = rewards_sim[t] + gamma * next_val - values_sim[t]
        last_gae = delta + gamma * lam * last_gae
        advantages_sim[t] = last_gae
    
    ax.plot(range(seq_len), advantages_sim, '-o', color=color, linewidth=2,
            markersize=4, label=f'λ={lam}')

ax.axvline(seq_len-1, color='black', linestyle='--', alpha=0.5, label='Terminal reward')
ax.set_xlabel('Token Position')
ax.set_ylabel('Advantage A_t^GAE')
ax.set_title(f'GAE Advantage Propagation (sparse reward at t={seq_len-1})')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Bias-Variance tradeoff with lambda
ax = axes[1]
lambdas = np.linspace(0, 1, 50)
# Simulated bias and variance (illustrative)
bias = np.exp(-5 * lambdas)  # decreases with lambda
variance = lambdas ** 2       # increases with lambda

ax.plot(lambdas, bias, 'r-', linewidth=2.5, label='Bias ↓ with λ')
ax.plot(lambdas, variance, 'b-', linewidth=2.5, label='Variance ↑ with λ')
ax.plot(lambdas, bias + variance, 'g--', linewidth=2, label='Total Error')

ax.axvline(0.95, color='green', linestyle='--', alpha=0.7, label='λ=0.95 (sweet spot)')
ax.set_xlabel('GAE Lambda (λ)')
ax.set_ylabel('Error')
ax.set_title('GAE Bias-Variance Tradeoff')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('gae_advantage.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. Full PPO Objective for Language Models

In [ ]:
def ppo_policy_loss(
    logprobs: torch.Tensor,
    old_logprobs: torch.Tensor,
    advantages: torch.Tensor,
    masks: torch.Tensor,
    epsilon: float = 0.2,
) -> Tuple[torch.Tensor, dict]:
    """
    PPO clipped policy loss.
    
    L_CLIP = E_t[ min(r_t * A_t, clip(r_t, 1-ε, 1+ε) * A_t) ]
    
    Args:
        logprobs:     Log probs under current policy π_θ (batch, seq)
        old_logprobs: Log probs under old policy π_θ_old (batch, seq)
        advantages:   GAE advantages (batch, seq)
        masks:        Valid token mask (batch, seq)
        epsilon:      Clip range
    
    Returns:
        loss: Scalar PPO policy loss
        metrics: Dictionary of diagnostic metrics
    """
    # Importance sampling ratio
    ratio = torch.exp(logprobs - old_logprobs)   # r_t(θ) = π_θ(a|s) / π_θ_old(a|s)
    
    # Unclipped and clipped surrogates
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages
    
    # Take minimum (conservative objective)
    policy_loss = -torch.min(surr1, surr2)
    
    # Apply mask and average over valid tokens
    policy_loss = (policy_loss * masks).sum() / masks.sum()
    
    # Fraction of time clipping is active (diagnostic)
    with torch.no_grad():
        is_clipped = ((ratio < 1 - epsilon) | (ratio > 1 + epsilon)).float()
        clip_fraction = (is_clipped * masks).sum() / masks.sum()
        approx_kl = 0.5 * ((logprobs - old_logprobs)**2 * masks).sum() / masks.sum()
    
    metrics = {
        'policy_loss': policy_loss.item(),
        'clip_fraction': clip_fraction.item(),
        'approx_kl': approx_kl.item(),
        'mean_ratio': (ratio * masks).sum().item() / masks.sum().item(),
    }
    return policy_loss, metrics


def ppo_value_loss(
    values: torch.Tensor,
    old_values: torch.Tensor,
    returns: torch.Tensor,
    masks: torch.Tensor,
    epsilon: float = 0.2,
) -> torch.Tensor:
    """
    PPO clipped value function loss.
    Clipping prevents large value function updates.
    """
    # Unclipped value loss
    value_loss_unclipped = (values - returns) ** 2
    
    # Clipped value function (value can't move too far from old)
    values_clipped = old_values + torch.clamp(values - old_values, -epsilon, epsilon)
    value_loss_clipped = (values_clipped - returns) ** 2
    
    # Take max (conservative)
    value_loss = 0.5 * torch.max(value_loss_unclipped, value_loss_clipped)
    return (value_loss * masks).sum() / masks.sum()


def compute_kl_penalty(
    logprobs: torch.Tensor,
    ref_logprobs: torch.Tensor,
    masks: torch.Tensor,
) -> torch.Tensor:
    """
    Per-token KL divergence between policy and reference model.
    KL(π_θ || π_ref) = log(π_θ/π_ref) (approximation for single token)
    """
    kl_per_token = logprobs - ref_logprobs
    return (kl_per_token * masks).sum(dim=-1) / masks.sum(dim=-1).clamp(min=1)


def compute_rlhf_rewards(
    scores: torch.Tensor,   # Reward model scores (batch,)
    logprobs: torch.Tensor, # Policy log probs (batch, seq)
    ref_logprobs: torch.Tensor, # Reference log probs (batch, seq)
    masks: torch.Tensor,    # Valid tokens (batch, seq)
    kl_coeff: float = 0.05,
) -> torch.Tensor:
    """
    Compute token-level RLHF rewards:
    r_t = -kl_coeff * (log π_θ(a_t) - log π_ref(a_t))  for t < T
    r_T = r_φ(x,y) - kl_coeff * KL_T                   for final token
    """
    batch_size, seq_len = masks.shape
    
    # Per-token KL penalty (negative reward for deviating from reference)
    kl_per_token = logprobs - ref_logprobs  # (batch, seq)
    token_rewards = -kl_coeff * kl_per_token
    
    # Add reward model score to the LAST valid token
    last_token_idx = masks.long().sum(dim=-1) - 1  # (batch,)
    for i, idx in enumerate(last_token_idx):
        token_rewards[i, idx] += scores[i]
    
    return token_rewards * masks


# Test RLHF reward computation
batch_size, seq_len = 3, 15
scores = torch.tensor([2.0, 0.5, -0.5])  # Reward model scores
logprobs = torch.randn(batch_size, seq_len) * 0.1 - 2.0
ref_logprobs = torch.randn(batch_size, seq_len) * 0.1 - 2.0
masks = torch.ones(batch_size, seq_len)
masks[0, 12:] = 0  # First example is shorter

rlhf_rewards = compute_rlhf_rewards(scores, logprobs, ref_logprobs, masks, kl_coeff=0.05)

print("RLHF Token-Level Rewards:")
for i in range(batch_size):
    valid_len = int(masks[i].sum())
    print(f"  Example {i} (score={scores[i]:.1f}, len={valid_len}): "
          f"mean_reward={rlhf_rewards[i, :valid_len].mean():.4f}, "
          f"last_token_reward={rlhf_rewards[i, valid_len-1]:.4f}")

In [ ]:
# Simulate a full PPO training loop to show dynamics
np.random.seed(42)

def simulate_ppo_training(
    n_steps: int = 100,
    epsilon: float = 0.2,
    kl_target: float = 0.01,
    initial_reward: float = -1.0,
):
    """
    Simulate PPO training dynamics for a language model task.
    Shows how reward improves while KL divergence is controlled.
    """
    rewards_history = []
    kl_history = []
    policy_loss_history = []
    value_loss_history = []
    clip_frac_history = []
    
    # Simulate improvement with noise
    reward = initial_reward
    kl = 0.0
    
    for step in range(n_steps):
        # Simulate reward improvement (logistic growth)
        target_reward = 2.0 * (1 / (1 + np.exp(-0.08 * (step - 50))))
        reward = 0.9 * reward + 0.1 * target_reward + np.random.randn() * 0.1
        
        # KL divergence: controlled by clipping
        # Goes up as policy improves, controlled to stay near target
        kl_target_local = kl_target * (1 + step / n_steps)  # Slowly increasing
        kl = 0.85 * kl + 0.15 * kl_target_local + np.random.randn() * 0.002
        kl = max(kl, 0)
        
        # Policy loss (should decrease) 
        pl = -reward * 0.1 + np.random.randn() * 0.05
        
        # Value loss (decreases as critic improves)
        vl = max(0.5 * np.exp(-0.02 * step) + np.random.randn() * 0.05, 0)
        
        # Clip fraction (should be low, ~5-15%)
        cf = 0.1 * (1 + kl / kl_target) + np.random.randn() * 0.02
        cf = np.clip(cf, 0, 0.5)
        
        rewards_history.append(reward)
        kl_history.append(kl)
        policy_loss_history.append(pl)
        value_loss_history.append(vl)
        clip_frac_history.append(cf)
    
    return {
        'rewards': rewards_history,
        'kl': kl_history,
        'policy_loss': policy_loss_history,
        'value_loss': value_loss_history,
        'clip_fraction': clip_frac_history,
    }


sim_data = simulate_ppo_training(n_steps=200)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
steps = list(range(len(sim_data['rewards'])))

# Smooth for better visualization
def smooth(x, w=10):
    kernel = np.ones(w) / w
    return np.convolve(x, kernel, mode='valid')

plot_configs = [
    ('rewards', 'Mean Reward', 'Reward Model Score', '#2ecc71', True),
    ('kl', 'KL Divergence from SFT', 'KL(π_θ || π_SFT)', '#e74c3c', False),
    ('policy_loss', 'Policy Loss', 'PPO Policy Loss', '#3498db', False),
    ('value_loss', 'Value Function Loss', 'Critic (MSE) Loss', '#9b59b6', False),
    ('clip_fraction', 'PPO Clip Fraction', 'Fraction of Clipped Updates', '#e67e22', False),
]

for ax_idx, (key, title, ylabel, color, add_ref) in enumerate(plot_configs):
    row, col = ax_idx // 3, ax_idx % 3
    ax = axes[row][col]
    
    raw = np.array(sim_data[key])
    smoothed = smooth(raw, w=10)
    
    ax.plot(steps, raw, color=color, alpha=0.3, linewidth=1)
    ax.plot(range(len(smoothed)), smoothed, color=color, linewidth=2.5)
    
    if add_ref:
        ax.axhline(0, color='red', linestyle='--', alpha=0.5, label='SFT baseline')
    
    ax.set_xlabel('PPO Steps')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    if add_ref:
        ax.legend()

# Plot 6: KL-Reward Pareto frontier
ax = axes[1][2]
rewards_arr = np.array(sim_data['rewards'])
kl_arr = np.array(sim_data['kl'])
scatter = ax.scatter(kl_arr, rewards_arr, c=steps, cmap='viridis', alpha=0.6, s=20)
plt.colorbar(scatter, ax=ax, label='Training Step')
ax.set_xlabel('KL Divergence from SFT')
ax.set_ylabel('Mean Reward')
ax.set_title('Reward vs KL Tradeoff (Training Trajectory)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ppo_training_dynamics.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Practical RLHF with TRL PPO Trainer

In [ ]:
# ============================================================
# RLHF Configuration
# ============================================================

# Model names
POLICY_MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"  # SFT model to align
REWARD_MODEL_NAME = "OpenAssistant/reward-model-deberta-v3-large-v2"  # Pre-trained RM
# Other reward models: "weqweasdas/RM-Mistral-7B", "Elyza/Elyza-tasks-1-reward"

DATASET_NAME = "HuggingFaceH4/ultrachat_200k"  # Prompts for generation

# PPO Hyperparameters
PPO_CONFIG = {
    # Core PPO
    "learning_rate": 1.41e-5,
    "mini_batch_size": 4,
    "batch_size": 16,                   # Rollout batch size
    "gradient_accumulation_steps": 1,
    "ppo_epochs": 4,                    # PPO update epochs per batch
    "cliprange": 0.2,                   # Epsilon for PPO clipping
    "cliprange_value": 0.2,             # Epsilon for value function clipping
    
    # GAE
    "gamma": 1.0,                       # Discount (1.0 for text gen)
    "lam": 0.95,                        # GAE lambda
    
    # KL penalty
    "init_kl_coef": 0.05,              # Initial KL coefficient
    "target_kl": 6.0,                   # Target KL (adaptive control)
    "horizon": 10000,
    
    # Value function
    "vf_coef": 0.1,                     # Value function loss coefficient
    
    # Generation
    "max_new_tokens": 256,
    "temperature": 0.7,
    "top_p": 0.9,
    "top_k": 0,
    
    # Training
    "num_ppo_epochs": 1,
    "total_episodes": 10000,
    "logging_steps": 10,
    "output_dir": "./ppo-rlhf-output",
    "report_to": "none",
}

print("RLHF PPO Configuration:")
print(f"  Policy model:   {POLICY_MODEL_NAME}")
print(f"  Reward model:   {REWARD_MODEL_NAME}")
print(f"  Dataset:        {DATASET_NAME}")
print(f"  PPO clip range: {PPO_CONFIG['cliprange']}")
print(f"  GAE lambda:     {PPO_CONFIG['lam']}")
print(f"  KL coefficient: {PPO_CONFIG['init_kl_coef']} (adaptive, target={PPO_CONFIG['target_kl']})")

In [ ]:
def train_with_rlhf_ppo():
    """
    Complete RLHF training pipeline using TRL PPOTrainer.
    
    Requires:
    - GPU with 24+ GB VRAM (for 1B policy + reward model)
    - HuggingFace model access
    """
    from transformers import (
        AutoModelForCausalLM,
        AutoModelForSequenceClassification,
        AutoTokenizer,
        pipeline,
    )
    from peft import LoraConfig
    from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead
    from datasets import load_dataset
    import torch
    
    # Step 1: Load policy model with value head
    print("Step 1: Loading policy model with value head...")
    # TRL's AutoModelForCausalLMWithValueHead adds a value head for PPO
    policy_model = AutoModelForCausalLMWithValueHead.from_pretrained(
        POLICY_MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        # Optional: add LoRA for parameter efficiency
        peft_config=LoraConfig(
            r=8,
            lora_alpha=16,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            task_type="CAUSAL_LM",
        ),
    )
    
    # Step 2: Load tokenizer
    print("Step 2: Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(POLICY_MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"  # Left padding for generation
    
    # Step 3: Load reward model
    print(f"Step 3: Loading reward model '{REWARD_MODEL_NAME}'...")
    # Use sentiment pipeline for convenience (returns scores)
    reward_pipeline = pipeline(
        model=REWARD_MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        truncation=True,
        max_length=512,
    )
    
    # Step 4: Load dataset (extract prompts only)
    print(f"Step 4: Loading dataset '{DATASET_NAME}'...")
    dataset = load_dataset(DATASET_NAME, split="train_sft", streaming=True)
    
    def format_prompt(example):
        """Format as instruction prompt."""
        messages = example.get('messages', [])
        if messages:
            return {"query": tokenizer.apply_chat_template(
                [messages[0]],  # Just the first user message
                tokenize=False,
                add_generation_prompt=True,
            )}
        return {"query": example.get("prompt", "")}
    
    # Step 5: Configure PPO
    print("Step 5: Setting up PPO trainer...")
    ppo_config = PPOConfig(
        output_dir=PPO_CONFIG["output_dir"],
        learning_rate=PPO_CONFIG["learning_rate"],
        mini_batch_size=PPO_CONFIG["mini_batch_size"],
        batch_size=PPO_CONFIG["batch_size"],
        gradient_accumulation_steps=PPO_CONFIG["gradient_accumulation_steps"],
        ppo_epochs=PPO_CONFIG["ppo_epochs"],
        cliprange=PPO_CONFIG["cliprange"],
        cliprange_value=PPO_CONFIG["cliprange_value"],
        gamma=PPO_CONFIG["gamma"],
        lam=PPO_CONFIG["lam"],
        init_kl_coef=PPO_CONFIG["init_kl_coef"],
        target_kl=PPO_CONFIG["target_kl"],
        vf_coef=PPO_CONFIG["vf_coef"],
        report_to=PPO_CONFIG["report_to"],
    )
    
    ppo_trainer = PPOTrainer(
        config=ppo_config,
        model=policy_model,
        tokenizer=tokenizer,
        dataset=dataset,
        data_collator=None,
    )
    
    # Step 6: PPO Training Loop
    print("Step 6: Starting RLHF training loop...")
    generation_kwargs = {
        "max_new_tokens": PPO_CONFIG["max_new_tokens"],
        "temperature": PPO_CONFIG["temperature"],
        "top_p": PPO_CONFIG["top_p"],
        "do_sample": True,
        "pad_token_id": tokenizer.eos_token_id,
    }
    
    for epoch, batch in enumerate(ppo_trainer.dataloader):
        # 1. Tokenize prompts
        query_tensors = batch["input_ids"]
        
        # 2. Generate responses (rollout)
        response_tensors = ppo_trainer.generate(
            query_tensors, return_prompt=False, **generation_kwargs
        )
        batch["response"] = tokenizer.batch_decode(response_tensors, skip_special_tokens=True)
        
        # 3. Score responses with reward model
        texts = [q + r for q, r in zip(batch["query"], batch["response"])]
        pipe_outputs = reward_pipeline(texts)
        
        # Extract scores (format depends on reward model output)
        rewards = []
        for output in pipe_outputs:
            if isinstance(output, list):
                score = max(output, key=lambda x: x['score'])['score']
            else:
                score = output['score']
            rewards.append(torch.tensor(score, dtype=torch.float))
        
        # 4. PPO update step
        stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
        ppo_trainer.log_stats(stats, batch, rewards)
        
        if epoch >= 100:  # Limit training steps for demo
            break
    
    # Step 7: Save the aligned model
    print("Step 7: Saving RLHF-aligned model...")
    ppo_trainer.save_pretrained(PPO_CONFIG["output_dir"])
    
    print("✓ RLHF PPO training complete!")
    return policy_model, tokenizer


# Uncomment to run training (requires GPU + HuggingFace access):
# policy_model, tokenizer = train_with_rlhf_ppo()

print("RLHF PPO training function defined.")
print("To run: uncomment the last line above (requires GPU).")

## 7. Monitoring RLHF Training

In [ ]:
# Key metrics to monitor during RLHF training
# (These are what you'd see in W&B / TensorBoard during real training)

np.random.seed(42)
n_steps = 300
steps = np.arange(n_steps)

# Simulate realistic RLHF training metrics
def simulate_metric(base, target, steps, noise_scale=0.1, warmup=20):
    progress = 1 / (1 + np.exp(-0.05 * (steps - warmup)))
    return base + (target - base) * progress + np.random.randn(len(steps)) * noise_scale

metrics = {
    'mean_reward': simulate_metric(-0.5, 1.8, steps, noise_scale=0.15),
    'kl_divergence': simulate_metric(0, 4.5, steps, noise_scale=0.2),
    'policy_entropy': simulate_metric(3.2, 2.6, steps, noise_scale=0.1),
    'value_loss': simulate_metric(0.8, 0.1, steps, noise_scale=0.05),
    'approx_kl': np.abs(simulate_metric(0, 0.01, steps, noise_scale=0.003)),
    'clip_fraction': np.clip(simulate_metric(0, 0.12, steps, noise_scale=0.03), 0, 0.5),
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

metric_configs = [
    ('mean_reward', 'Mean Reward', 'Avg reward from RM', '#2ecc71'),
    ('kl_divergence', 'KL Divergence', 'KL(π_θ || π_SFT)', '#e74c3c'),
    ('policy_entropy', 'Policy Entropy', 'Entropy of token distribution', '#3498db'),
    ('value_loss', 'Value Loss', 'Critic MSE loss', '#9b59b6'),
    ('approx_kl', 'Approx KL per update', 'PPO step KL', '#e67e22'),
    ('clip_fraction', 'Clip Fraction', 'Fraction of clipped updates', '#1abc9c'),
]

def smooth_curve(arr, w=15):
    return np.convolve(arr, np.ones(w)/w, mode='valid')

for idx, (key, title, ylabel, color) in enumerate(metric_configs):
    row, col = idx // 3, idx % 3
    ax = axes[row][col]
    
    raw = metrics[key]
    smoothed = smooth_curve(raw)
    smooth_steps = steps[:len(smoothed)]
    
    ax.plot(steps, raw, color=color, alpha=0.3, linewidth=1)
    ax.plot(smooth_steps, smoothed, color=color, linewidth=2.5, label='Smoothed')
    ax.set_xlabel('PPO Training Steps')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

# Add reference lines
axes[0][0].axhline(0, color='red', linestyle='--', alpha=0.5, label='SFT baseline (≈0)')
axes[0][0].legend()
axes[0][1].axhline(6, color='orange', linestyle='--', alpha=0.7, label='KL target=6')
axes[0][1].legend()
axes[1][0].axhline(0.1, color='green', linestyle='--', alpha=0.7, label='Target clip < 10%')
axes[1][0].legend()

plt.suptitle('RLHF PPO Training Dashboard', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('rlhf_ppo_dashboard.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Reward Hacking and Mitigation

In [ ]:
# Visualize reward hacking phenomenon
np.random.seed(42)
n_steps = 500
steps = np.arange(n_steps)

# Reward hacking: RM score goes up but true quality eventually plateaus/decreases
def reward_hack_sim(steps, peak_step=200, hack_start=150):
    rm_score = np.zeros(len(steps))
    true_quality = np.zeros(len(steps))
    
    for i, t in enumerate(steps):
        # RM score: keeps going up (overfitting to RM)
        rm_score[i] = 3.0 * (1 - np.exp(-0.015 * t)) + 0.001 * t + np.random.randn() * 0.1
        # True quality: peaks and degrades (reward hacking)
        true_quality[i] = (2.0 * (1 - np.exp(-0.02 * t)) - 
                          0.005 * max(0, t - hack_start) + np.random.randn() * 0.1)
    return rm_score, true_quality

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Reward hacking illustration
ax = axes[0]
rm_score, true_quality = reward_hack_sim(steps)
ax.plot(steps, np.convolve(rm_score, np.ones(15)/15, 'valid'), 'r-', 
        linewidth=2.5, label='RM Score (proxy reward)')
ax.plot(steps[:len(np.convolve(true_quality, np.ones(15)/15, 'valid'))], 
        np.convolve(true_quality, np.ones(15)/15, 'valid'), 
        'b-', linewidth=2.5, label='True Quality (human eval)')
ax.axvline(150, color='gray', linestyle='--', alpha=0.7, label='Reward hacking begins')
ax.fill_between(steps, rm_score, color='red', alpha=0.1)
ax.set_xlabel('PPO Training Steps')
ax.set_ylabel('Score')
ax.set_title('Reward Hacking: RM Score vs True Quality')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: KL as regularization
ax = axes[1]
kl_coefficients = [0.0, 0.02, 0.05, 0.1, 0.2]
colors = plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(kl_coefficients)))

for beta, color in zip(kl_coefficients, colors):
    # Simulate peak quality with different KL coefficients
    # Higher beta → less reward hacking, but also less reward improvement
    peak_quality = 2.5 * (1 - np.exp(-5 * beta)) / (1 + 2 * beta)
    hack_decay = 0.005 / (1 + 10 * beta)  # Less hacking with higher beta
    
    quality_curve = np.array([
        peak_quality * (1 - np.exp(-0.02*t)) - hack_decay * max(0, t-150) + np.random.randn() * 0.05
        for t in steps
    ])
    quality_smooth = np.convolve(quality_curve, np.ones(20)/20, 'valid')
    ax.plot(range(len(quality_smooth)), quality_smooth, color=color, linewidth=2,
            label=f'β={beta}')

ax.axvline(150, color='gray', linestyle='--', alpha=0.7)
ax.set_xlabel('PPO Training Steps')
ax.set_ylabel('True Quality Score')
ax.set_title('KL Penalty β Controls Reward Hacking')
ax.legend(title='KL coeff β')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reward_hacking.png', dpi=100, bbox_inches='tight')
plt.show()

print("Key insight: KL penalty (β) is critical for preventing reward hacking.")
print("Too small β → reward hacking; Too large β → model stays near SFT (no alignment improvement)")

## 9. Summary: The Complete RLHF Stack

### Full PPO Algorithm for Language Models

```
Initialize: π_θ (policy), V_ψ (critic), r_φ (frozen reward model), π_SFT (frozen reference)

For each PPO batch:
  ─── ROLLOUT PHASE ───
  1. Sample prompts x from dataset
  2. Generate responses y ~ π_θ(y|x)  (old policy)
  3. Compute rewards: r_t = r_φ(x,y) - β·KL(π_θ(y_t|s_t) || π_SFT(y_t|s_t))
  4. Compute values: V_t = V_ψ(s_t)  for each token
  5. Compute GAE advantages: A_t = Σ (γλ)^l δ_{t+l}
  
  ─── UPDATE PHASE (repeat for ppo_epochs) ───
  For each mini-batch:
    6. Compute policy log probs under new π_θ
    7. Policy loss:  L_clip = E[min(r_t·A_t, clip(r_t, 1±ε)·A_t)]
    8. Value loss:   L_VF = E[max((V-R)², (clip(V,V_old±ε)-R)²)]
    9. Total loss:   L = -L_clip + c_v·L_VF - c_e·H(π_θ)
    10. Update θ, ψ via gradient descent
    11. If approx_KL > target_KL: break (early stopping)
```

### PPO Key Hyperparameters

| Parameter | Recommended | Effect |
|---|---|---|
| `cliprange` ε | 0.2 | Policy update size limit |
| `ppo_epochs` | 4 | Reuse each rollout N times |
| `gae_lambda` λ | 0.95 | Advantage estimation bias-variance |
| `init_kl_coef` β | 0.02 – 0.1 | KL penalty weight |
| `target_kl` | 1 – 10 | Adaptive KL control target |
| `vf_coef` | 0.1 | Value function loss weight |
| `learning_rate` | ~1e-5 | Very small for stability |
| `mini_batch_size` | 4-8 | Mini-batches for PPO updates |

### RLHF Failure Modes and Mitigations

| Issue | Symptom | Fix |
|---|---|---|
| **Reward hacking** | RM score ↑, quality ↓ | Increase KL penalty β |
| **Mode collapse** | Low entropy, repetitive outputs | Add entropy bonus |
| **Training instability** | Loss spikes, divergence | Reduce LR, reduce `cliprange` |
| **KL explosion** | KL >> target_kl | Early stop per PPO epoch |
| **Value function lag** | High value loss | More value updates, higher `vf_coef` |

### DPO vs PPO Summary

| | DPO | PPO (RLHF) |
|---|---|---|
| Reward model needed? | No | Yes |
| Online generation? | No | Yes |
| Number of models | 2 | 4 |
| Compute | Low | High |
| Implementation | Simple | Complex |
| Best for | Offline alignment, limited resources | Maximum performance, online feedback |